# NSL-KDD Data Preprocessing

This notebook prepares the NSL-KDD training and test datasets for intrusion-detection models. It loads the raw CSV files with their 43 feature, label, and difficulty columns, then inspects dataset structure, feature types, missing values, duplicate rows, and normal-versus-attack distributions. It also identifies attack types found in the test set but absent from training.

For modeling, the notebook separates the 41 input features from the attack label and difficulty metadata, creates binary labels (`normal` = 0, attack = 1), and identifies categorical and numerical features. It builds a preprocessing pipeline that one-hot encodes categorical features and standardizes numerical features. The pipeline is fitted only on training data and then applied to the test data, with matching processed feature dimensions checked to avoid data leakage and ensure consistency.

Finally, the fitted preprocessor is saved to `models/preprocessor.joblib` so subsequent modeling notebooks can reuse the same transformations.

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path

# Resolve the dataset directory from either the project root or notebooks/.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "dataset" / "raw"

TRAIN_PATH = DATA_DIR / "KDDTrain+.csv"
TEST_PATH = DATA_DIR / "KDDTest+.csv"


In [6]:
# Name the 41 input features in the official dataset order
columns = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins",
    "logged_in", "num_compromised", "root_shell", "su_attempted",
    "num_root", "num_file_creations", "num_shells", "num_access_files",
    "num_outbound_cmds", "is_host_login", "is_guest_login", "count",
    "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate",
    "srv_rerror_rate", "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate",
    "dst_host_count", "dst_host_srv_count", "dst_host_same_srv_rate",
    "dst_host_diff_srv_rate", "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate", "dst_host_serror_rate",
    "dst_host_srv_serror_rate", "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate", "attack", "difficulty"
]

# Check that the column definition is complete
print(len(columns))
assert len(columns) == 43


43


In [14]:
# The raw NSL-KDD files generally have no header row
train_df = pd.read_csv(TRAIN_PATH, header=None, names=columns)
test_df = pd.read_csv(TEST_PATH, header=None, names=columns)

# Check dataset dimensions
print("Training shape:", train_df.shape)
print("Test shape:", test_df.shape)

display(train_df.head())

Training shape: (125973, 43)
Test shape: (22543, 43)


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,attack,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


In [15]:
# Inspect column types and non-null counts
train_df.info()

# Summarize numerical features
display(train_df.describe().T)

# Inspect categorical feature values
for col in ["protocol_type", "service", "flag"]:
    print(f"\n{col}: {train_df[col].nunique()} unique values")
    print(train_df[col].value_counts().head(10))

<class 'pandas.DataFrame'>
RangeIndex: 125973 entries, 0 to 125972
Data columns (total 43 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   duration                     125973 non-null  int64  
 1   protocol_type                125973 non-null  str    
 2   service                      125973 non-null  str    
 3   flag                         125973 non-null  str    
 4   src_bytes                    125973 non-null  int64  
 5   dst_bytes                    125973 non-null  int64  
 6   land                         125973 non-null  int64  
 7   wrong_fragment               125973 non-null  int64  
 8   urgent                       125973 non-null  int64  
 9   hot                          125973 non-null  int64  
 10  num_failed_logins            125973 non-null  int64  
 11  logged_in                    125973 non-null  int64  
 12  num_compromised              125973 non-null  int64  
 13  root_shell

,count,mean,std,min,25%,50%,75%,max
duration,125973.0,287.144650,2.604515e+03,0.0,0.00,0.00,0.00,4.290800e+04
src_bytes,125973.0,45566.743000,5.870331e+06,0.0,0.00,44.00,276.00,1.379964e+09
dst_bytes,125973.0,19779.114421,4.021269e+06,0.0,0.00,0.00,516.00,1.309937e+09
land,125973.0,0.000198,1.408607e-02,0.0,0.00,0.00,0.00,1.000000e+00
wrong_fragment,125973.0,0.022687,2.535300e-01,0.0,0.00,0.00,0.00,3.000000e+00
urgent,125973.0,0.000111,1.436603e-02,0.0,0.00,0.00,0.00,3.000000e+00
hot,125973.0,0.204409,2.149968e+00,0.0,0.00,0.00,0.00,7.700000e+01
num_failed_logins,125973.0,0.001222,4.523914e-02,0.0,0.00,0.00,0.00,5.000000e+00
logged_in,125973.0,0.395736,4.890101e-01,0.0,0.00,0.00,1.00,1.000000e+00
num_compromised,125973.0,0.279250,2.394204e+01,0.0,0.00,0.00,0.00,7.479000e+03



protocol_type: 3 unique values
protocol_type
tcp     102689
udp      14993
icmp      8291
Name: count, dtype: int64

service: 70 unique values
service
http        40338
private     21853
domain_u     9043
smtp         7313
ftp_data     6860
eco_i        4586
other        4359
ecr_i        3077
telnet       2353
finger       1767
Name: count, dtype: int64

flag: 11 unique values
flag
SF        74945
S0        34851
REJ       11233
RSTR       2421
RSTO       1562
S1          365
SH          271
S2          127
RSTOS0      103
S3           49
Name: count, dtype: int64


In [16]:
# Check missing values in both datasets
print("Training missing values:", train_df.isnull().sum().sum())
print("Test missing values:", test_df.isnull().sum().sum())

# Count duplicate rows without removing them yet
print("Training duplicate rows:", train_df.duplicated().sum())
print("Test duplicate rows:", test_df.duplicated().sum())

Training missing values: 0
Test missing values: 0
Training duplicate rows: 0
Test duplicate rows: 0


In [17]:
# Count normal and malicious connections
print("Training labels:")
display(train_df["attack"].value_counts())

print("Test labels:")
display(test_df["attack"].value_counts())

# Compare normal versus attack proportions
print("Training proportions:")
display(train_df["attack"].value_counts(normalize=True).mul(100).round(2))

print("Test proportions:")
display(test_df["attack"].value_counts(normalize=True).mul(100).round(2))

Training labels:


attack
normal             67343
neptune            41214
satan               3633
ipsweep             3599
portsweep           2931
smurf               2646
nmap                1493
back                 956
teardrop             892
warezclient          890
pod                  201
guess_passwd          53
buffer_overflow       30
warezmaster           20
land                  18
imap                  11
rootkit               10
loadmodule             9
ftp_write              8
multihop               7
phf                    4
perl                   3
spy                    2
Name: count, dtype: int64

Test labels:


attack
normal             9710
neptune            4657
guess_passwd       1231
mscan               996
warezmaster         944
apache2             737
satan               735
processtable        685
smurf               665
back                359
snmpguess           331
saint               319
mailbomb            293
snmpgetattack       178
portsweep           157
ipsweep             141
httptunnel          133
nmap                 73
pod                  41
buffer_overflow      20
multihop             18
named                17
ps                   15
sendmail             14
xterm                13
rootkit              13
teardrop             12
xlock                 9
land                  7
xsnoop                4
ftp_write             3
loadmodule            2
worm                  2
perl                  2
sqlattack             2
udpstorm              2
phf                   2
imap                  1
Name: count, dtype: int64

Training proportions:


attack
normal             53.46
neptune            32.72
satan               2.88
ipsweep             2.86
portsweep           2.33
smurf               2.10
nmap                1.19
back                0.76
teardrop            0.71
warezclient         0.71
pod                 0.16
guess_passwd        0.04
buffer_overflow     0.02
warezmaster         0.02
land                0.01
imap                0.01
rootkit             0.01
loadmodule          0.01
ftp_write           0.01
multihop            0.01
phf                 0.00
perl                0.00
spy                 0.00
Name: proportion, dtype: float64

Test proportions:


attack
normal             43.07
neptune            20.66
guess_passwd        5.46
mscan               4.42
warezmaster         4.19
apache2             3.27
satan               3.26
processtable        3.04
smurf               2.95
back                1.59
snmpguess           1.47
saint               1.42
mailbomb            1.30
snmpgetattack       0.79
portsweep           0.70
ipsweep             0.63
httptunnel          0.59
nmap                0.32
pod                 0.18
buffer_overflow     0.09
multihop            0.08
named               0.08
ps                  0.07
sendmail            0.06
xterm               0.06
rootkit             0.06
teardrop            0.05
xlock               0.04
land                0.03
xsnoop              0.02
ftp_write           0.01
loadmodule          0.01
worm                0.01
perl                0.01
sqlattack           0.01
udpstorm            0.01
phf                 0.01
imap                0.00
Name: proportion, dtype: float64

In [18]:
# Find attack types appearing in the test set but not the training set
train_attacks = set(train_df["attack"].unique())
test_attacks = set(test_df["attack"].unique())

unseen_attacks = sorted(test_attacks - train_attacks)

print("Attack types absent from training:", unseen_attacks)
print("Number of unseen attack types:", len(unseen_attacks))

Attack types absent from training: ['apache2', 'httptunnel', 'mailbomb', 'mscan', 'named', 'processtable', 'ps', 'saint', 'sendmail', 'snmpgetattack', 'snmpguess', 'sqlattack', 'udpstorm', 'worm', 'xlock', 'xsnoop', 'xterm']
Number of unseen attack types: 17


In [19]:
# Exclude the target label and difficulty metadata from model inputs
X_train_raw = train_df.drop(columns=["attack", "difficulty"])
X_test_raw = test_df.drop(columns=["attack", "difficulty"])

y_train = train_df["attack"]
y_test = test_df["attack"]

# Convert labels into the primary binary classification task
y_train_binary = (y_train != "normal").astype(int)
y_test_binary = (y_test != "normal").astype(int)

print("Feature count:", X_train_raw.shape[1])
print("Training binary labels:\n", y_train_binary.value_counts())

Feature count: 41
Training binary labels:
 attack
0    67343
1    58630
Name: count, dtype: int64


In [20]:
# Identify categorical and numerical input columns
categorical_cols = X_train_raw.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_cols = X_train_raw.select_dtypes(
    exclude=["object"]
).columns.tolist()

print("Categorical features:", categorical_cols)
print("Number of categorical features:", len(categorical_cols))
print("Number of numerical features:", len(numerical_cols))

Categorical features: ['protocol_type', 'service', 'flag']
Number of categorical features: 3
Number of numerical features: 38


/tmp/ipykernel_434633/1814769561.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train_raw.select_dtypes(
/tmp/ipykernel_434633/1814769561.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are excluded by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `exclude` to remove them, or to `include` to keep them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to w

In [21]:
# Check whether the test set contains categories absent from training
for col in categorical_cols:
    train_values = set(X_train_raw[col].unique())
    test_values = set(X_test_raw[col].unique())

    unseen_values = test_values - train_values

    print(f"{col}: {len(unseen_values)} unseen categories")
    if unseen_values:
        print(" ", unseen_values)

protocol_type: 0 unseen categories
service: 0 unseen categories
flag: 0 unseen categories


In [22]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Encode categorical values and scale numerical features
categorical_pipeline = OneHotEncoder(
    handle_unknown="ignore"
)

numerical_pipeline = StandardScaler()

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", categorical_pipeline, categorical_cols),
        ("numerical", numerical_pipeline, numerical_cols),
    ]
)

print("Preprocessing pipeline created.")

Preprocessing pipeline created.


In [23]:
# Learn encoding categories and scaling statistics from training data only
X_train_processed = preprocessor.fit_transform(X_train_raw)

# Apply the learned transformations without refitting on test data
X_test_processed = preprocessor.transform(X_test_raw)

print("Processed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

# Verify that both datasets have identical feature dimensions
assert X_train_processed.shape[1] == X_test_processed.shape[1]

Processed training shape: (125973, 122)
Processed test shape: (22543, 122)


In [ ]:
#Save the fitted preprocessor
import joblib

from pathlib import Path

# Save the fitted pipeline so future notebooks can reuse it
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(preprocessor, MODEL_DIR / "preprocessor.joblib")

print("Preprocessor saved.")

Preprocessor saved.
